# Settings

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
import torch
import torch.nn.functional as F
from torch.utils.data import random_split
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_undirected
import torch.serialization
from torch_geometric.data.data import Data, DataEdgeAttr

from GCN import GCN


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device



device(type='cuda')

# Encoder

It takes in the dataset with the trace featrures and produces a `µ`, `logvar`, and `z`

In [29]:
class GraphEncoderVAE(nn.Module):
    """
    Graph encoder for a VAE:
    - Takes a trace graph (x, edge_index, batch)
    - Outputs latent parameters (mu, logvar) and a sampled latent vector z
    """
    def __init__ (self, n_pods, n_ops, duration_mean, duration_std, hidden_ch=128, embed_dim=48, latent_dim=64, dropout=0.35):
        super().__init__()
        
        self.pod_embeddings = nn.Embedding(n_pods, embed_dim)
        self.op_embeddings = nn.Embedding(n_ops, embed_dim)
        self.duration_encoder = nn.Sequential(
            nn.Linear(1, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )
        
        in_channels = embed_dim * 3
        self.conv1 = GCNConv(in_channels, hidden_ch)
        self.bn1 = nn.BatchNorm1d(hidden_ch)
        self.conv2 = GCNConv(hidden_ch,hidden_ch)
        self.bn2 = nn.BatchNorm1d(hidden_ch)
        
         # Graph-level representation size (same as before)
        self.lin_feat = nn.Linear(hidden_ch, hidden_ch// 2)
        self.dropout = dropout

        # --- VAE latent heads ---
        self.latent_dim = latent_dim
        self.mu_lin = nn.Linear(hidden_ch // 2, latent_dim)
        self.logvar_lin = nn.Linear(hidden_ch // 2, latent_dim)

        # duration normalization as buffers
        duration_std = max(duration_std, 1e-6)
        self.register_buffer("duration_mean", torch.tensor(duration_mean))
        self.register_buffer("duration_std", torch.tensor(duration_std))
    
    def build_feats(self, x):
        """
        Same feature construction as your GCN:
        - x[:,0] = pod id (categorical)
        - x[:,1] = op id  (categorical)
        - x[:,2] = duration (continuous)
        """
        pod_ids = x[:, 0].long().clamp(min=0, max=self.pod_embeddings.num_embeddings - 1)
        op_ids = x[:, 1].long().clamp(min=0, max=self.op_embeddings.num_embeddings - 1)

        duration = torch.clamp(x[:,2], min=0)
        duration = torch.log1p(duration)
        duration = (duration - self.duration_mean) / (self.duration_std + 1e-9)
        duration = duration.unsqueeze(-1)

        pod_feat = self.pod_embeddings(pod_ids)
        op_feat = self.op_embeddings(op_ids)
        duration_feat = self.duration_encoder(duration)

        return torch.cat([pod_feat, op_feat, duration_feat], dim=1)
    
    def encode_graph(self, x, edge_idx, batch):
        """
        Core GNN encoder: returns a graph-level feature vector h_graph.
        """
        feats = self.build_feats(x)
        
        h = self.conv1(feats, edge_idx)
        h = self.bn1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.conv2(h,edge_idx)
        h = self.bn2(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        # global graph representation
        h_graph = global_mean_pool(h, batch)  # [num_graphs, hidden_channels]
        h_graph = F.relu(self.lin_feat(h_graph))  # [num_graphs, hidden_channels//2]
        h_graph = F.dropout(h_graph, p=self.dropout, training=self.training)
        return h_graph
    
    def forward(self, x, edge_idx, batch):
        """
        Forward pass:
        returns (mu, logvar, z) for each graph in the batch.
        """
        h_graph = self.encode_graph(x, edge_idx, batch)
        
        mu = self.mu_lin(h_graph)
        logvar = self.logvar_lin(h_graph)

        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            z = mu + eps * std
        else:
            z = mu
        
        return mu, logvar, z

    
    def encode(self, x, edge_idx, batch):
        """
        Convenience method: same as forward, but more explicit name.
        """ 
        
        return self.forward(x, edge_idx, batch)
        


# Load Data 

In [30]:
class LoadDataset(InMemoryDataset):
    def __init__(self, datapath="./data.pt") -> None:
        torch.serialization.add_safe_globals([Data, DataEdgeAttr])
        super().__init__(".")
        data, slices = torch.load(datapath, weights_only=False)
        self.data, self.slices = data, slices
        x = data.x
        self.num_pods = int(x[:, 0].max().item()) + 1
        self.num_ops = int(x[:, 1].max().item()) + 1
        log_duration = torch.log1p(torch.clamp(x[:, 2], min=0))
        self.duration_mean = log_duration.mean().item()
        self.duration_std = log_duration.std().item() or 1.0

    def get(self, idx):
        data = super().get(idx)
        data.edge_index = to_undirected(data.edge_index)
        return data

In [31]:
def summarize_dataset(dataset):

    # num_graphs = data["y"].size(0)
    print("=== DATASET SUMMARY ===")
    print(f"Total graphs: {len(dataset)}")

    labels = []
    total_nodes = 0

    for data in dataset:
        if hasattr(data, "y") and data.y is not None:
            if data.y.numel() == 1:
                labels.append(int(data.y.item()))
            else:
                labels.extend(data.y.tolist())

        total_nodes += data.num_nodes

    if len(labels) == 0:
        print("No Labels found in dataset")
        return

    y = torch.tensor(labels)
    unique_labels, counts = torch.unique(y, return_counts=True)

    print("\nLabel distribution:")
    for label, count in zip(unique_labels.tolist(), counts.tolist()):
        pct = 100.0 * count / len(y)
        print(f"  Label {label}: {count} graphs ({pct:.1f}%)")

    print(f"\nNumber of unique labels: {len(unique_labels)}")
    print(f"Total nodes across all graphs: {total_nodes}")
    print(f"Average nodes per graph: {total_nodes / len(dataset):.2f}")


def split_dataset(dataset, train_ratio=0.8, seed=42):
    """
    Safely split a dataset into train and validation sets.
    Returns (train_dataset, val_dataset)
    """
    n_total = len(dataset)
    if n_total < 2:
        raise ValueError("Dataset must contain at least two graphs to split.")

    train_len = int(n_total * train_ratio)
    val_len = n_total - train_len  # ensures total matches exactly

    # Fix edge cases
    if train_len == 0:
        train_len = 1
        val_len = n_total - 1
    elif val_len == 0:
        val_len = 1
        train_len = n_total - 1

    generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = random_split(
        dataset, [train_len, val_len], generator=generator
    )

    return train_dataset, val_dataset


def create_dataloaders(train_dataset, val_dataset, batch_size=32, num_workers=0):
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    print(f"Dataloaders ready - batch size: {batch_size}")
    return train_loader, val_loader



In [35]:
def main():
    datapath = "./data.pt"
    dataset = LoadDataset(datapath)
    summarize_dataset(dataset)
    print(f"Unique Pods: {dataset.num_pods}, Unique Operations: {dataset.num_ops}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    encoder = GraphEncoderVAE(n_pods=dataset.num_pods, n_ops=dataset.num_ops, duration_mean=dataset.duration_mean, duration_std=dataset.duration_std, hidden_ch=128, embed_dim=48, latent_dim=64).to(device)
    
    loader = DataLoader(dataset, batch_size=8, shuffle=True)
    batch = next(iter(loader)).to(device)

    mu, logvar, z = encoder(batch.x, batch.edge_index, batch.batch)
    
    print(f"Mu Shape: {mu}]"
          f"\nLogVar Shape: {logvar.shape}"
          f"\nZ-shape: {z.shape}")
    
    return mu, logvar, z

mu, logvar, z = main()


=== DATASET SUMMARY ===
Total graphs: 2097

Label distribution:
  Label 0: 1089 graphs (51.9%)
  Label 1: 1008 graphs (48.1%)

Number of unique labels: 2
Total nodes across all graphs: 106020
Average nodes per graph: 50.56
Unique Pods: 28, Unique Operations: 200
Mu Shape: tensor([[-0.1799, -0.1904, -0.0902,  0.0156,  0.1452, -0.0714,  0.2612, -0.1485,
         -0.0234, -0.1476,  0.0574,  0.1362, -0.1120, -0.0211,  0.1923, -0.1023,
         -0.0359, -0.0100, -0.1403,  0.2592, -0.0075,  0.1071, -0.3018,  0.0351,
         -0.1062,  0.0918,  0.0405,  0.2926, -0.1149,  0.0877,  0.2602, -0.0426,
         -0.0048,  0.1498,  0.1833,  0.0170,  0.0245, -0.1279,  0.1990,  0.0931,
         -0.0360, -0.1288,  0.0437,  0.2595, -0.0385,  0.0406, -0.0300, -0.2419,
          0.0282,  0.1676, -0.2442, -0.0737, -0.0486,  0.0797, -0.1606,  0.2049,
          0.1183,  0.0987,  0.0980, -0.0098,  0.2231,  0.0224,  0.0009,  0.0448],
        [-0.1503, -0.0878,  0.0413,  0.0582, -0.1441, -0.1517,  0.2246, -0.040

In [37]:
z.device

device(type='cuda', index=0)

# Decoder 

Takes the parameters from the encoder and produces a synthetic graph using `VRDAG`

In [64]:
class GraphDecoderVRDAG(nn.Module):

    def __init__(self, latent_dim, n_pods, n_ops,
                 max_nodes=64, node_hidden_dim=128):
        super().__init__()

        self.latent_dim = latent_dim
        self.n_pods = n_pods
        self.n_ops = n_ops
        self.max_nodes = max_nodes
        self.node_hidden_dim = node_hidden_dim

        # Predict node count
        self.node_count_mlp = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, max_nodes)
        )

        # Positional embeddings
        self.pos_embeddings = nn.Embedding(max_nodes, node_hidden_dim)

        # Node hidden-state generator
        self.node_mlp = nn.Sequential(
            nn.Linear(latent_dim + node_hidden_dim, node_hidden_dim),
            nn.ReLU(),
            nn.Linear(node_hidden_dim, node_hidden_dim),
            nn.ReLU()
        )

        # Heads for node prediction
        self.pod_head = nn.Linear(node_hidden_dim, n_pods)
        self.op_head = nn.Linear(node_hidden_dim, n_ops)
        self.duration_head = nn.Linear(node_hidden_dim, 2)  # mean, logvar

        # Edge scorer
        self.edge_score = nn.Bilinear(node_hidden_dim, node_hidden_dim, 1)

    # -------------------------------------------------------
    def sample_n_nodes(self, z, deterministic=False):
        """Predict number of nodes."""
        if z.dim() == 1:
            z = z.unsqueeze(0)

        logits = self.node_count_mlp(z)
        probs = F.softmax(logits, dim=-1)

        if deterministic:
            idx = torch.argmax(probs, dim=-1).item()
        else:
            idx = torch.distributions.Categorical(probs).sample().item()

        n_nodes = idx + 1
        return max(1, min(self.max_nodes, n_nodes))

    # -------------------------------------------------------
    def decode_single(self, z, n_nodes=None, deterministic=False):
        """Decode a single sample from latent z."""

        # Ensure z is 1D
        if z.dim() == 2 and z.size(0) == 1:
            z = z.squeeze(0)
        assert z.dim() == 1

        device = z.device

        # Decide number of nodes
        if n_nodes is None:
            n_nodes = self.sample_n_nodes(z, deterministic)
        n_nodes = max(1, min(self.max_nodes, int(n_nodes)))

        # Build node inputs
        positions = torch.arange(n_nodes, device=device)
        pos_emb = self.pos_embeddings(positions)

        # Broadcast z to each node
        z_broadcast = z.unsqueeze(0).expand(n_nodes, -1)
        node_input = torch.cat([z_broadcast, pos_emb], dim=-1)

        # Generate node hidden states (FIXED!)
        node_hidden = self.node_mlp(node_input)

        # --- Predict node features ---
        # pod
        pod_logits = self.pod_head(node_hidden)
        pod_probs = F.softmax(pod_logits, dim=-1)
        pod_ids = (torch.argmax(pod_probs, dim=-1)
                   if deterministic else torch.distributions.Categorical(pod_probs).sample())

        # op
        op_logits = self.op_head(node_hidden)
        op_probs = F.softmax(op_logits, dim=-1)
        op_ids = (torch.argmax(op_probs, dim=-1)
                  if deterministic else torch.distributions.Categorical(op_probs).sample())

        # duration (Gaussian)
        duration_params = self.duration_head(node_hidden)
        mean = duration_params[:, 0]
        logvar = duration_params[:, 1]
        std = torch.exp(0.5 * logvar)

        if deterministic:
            log_dur = mean
        else:
            eps = torch.randn_like(std)
            log_dur = mean + eps * std

        duration = torch.expm1(log_dur).clamp(min=0.0)

        # --- Generate DAG edges ---
        edge_index_list = []

        for i in range(1, n_nodes):
            h_i = node_hidden[i].unsqueeze(0)
            h_prev = node_hidden[:i]

            h_i_exp = h_i.expand(i, -1)
            scores = self.edge_score(h_i_exp, h_prev).squeeze(-1)
            edge_probs = torch.sigmoid(scores)

            if deterministic:
                edge_mask = edge_probs > 0.5
            else:
                edge_mask = torch.bernoulli(edge_probs).bool()

            candidates = torch.arange(i, device=device)
            js = candidates[edge_mask]

            if js.numel() > 0:
                src = js
                dst = torch.full((js.numel(),), i, dtype=torch.long, device=device)
                edge_index_list.append(torch.stack([src, dst], dim=0))

        if len(edge_index_list) > 0:
            edge_index = torch.cat(edge_index_list, dim=1)
        else:
            edge_index = torch.empty((2, 0), dtype=torch.long, device=device)

        # --- Build PyG x ---
        x = torch.stack([
            pod_ids.float(),
            op_ids.float(),
            duration.float()
        ], dim=-1)

        return Data(x=x, edge_index=edge_index)


In [73]:
from torch_geometric.loader import DataLoader

def test_generation():
    datapath = "./data.pt"
    dataset = LoadDataset(datapath)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1) Create encoder + decoder
    latent_dim = 64
    encoder = GraphEncoderVAE(
        n_pods=dataset.num_pods,
        n_ops=dataset.num_ops,
        duration_mean=dataset.duration_mean,
        duration_std=dataset.duration_std,
        hidden_ch=128,
        embed_dim=48,
        latent_dim=latent_dim,
    ).to(device)

    decoder = GraphDecoderVRDAG(
        latent_dim=latent_dim,
        n_pods=dataset.num_pods,
        n_ops=dataset.num_ops,
        max_nodes=64,
        node_hidden_dim=128,
    ).to(device)

    # 2) Take a real graph, encode it
    loader = DataLoader(dataset, batch_size=1, shuffle=True)
    real_batch = next(iter(loader)).to(device)

    mu, logvar, z = encoder(real_batch.x, real_batch.edge_index, real_batch.batch)
    # z: [1, latent_dim] -> use first
    z_single = z[0]  # [latent_dim]

    # 3) Decode a synthetic graph
    synthetic_graph = decoder.decode_single(z_single, deterministic=False)

    print("Synthetic graph:")
    print("  num_nodes:", synthetic_graph.num_nodes)
    print("  num_edges:", synthetic_graph.num_edges)
    print("  x shape:", synthetic_graph.x.shape)
    print("  edge_index shape:", synthetic_graph.edge_index.shape)

    return synthetic_graph


In [75]:
synthetic_graph = test_generation()

Synthetic graph:
  num_nodes: 61
  num_edges: 863
  x shape: torch.Size([61, 3])
  edge_index shape: torch.Size([2, 863])


# Visualize Synethic Graph

Displays the synthetic graph with nodes and edges

# Training 

Trains the encoder-decoder architecture over epochs for better results

# Evaluation

Eval the synthetic graphs